In [ ]:
import cProfile
import pstats
import io
import time
import traceback
import os, time
import csv
import numpy as np
from pathlib import Path
from datetime import timedelta
from fastai.callback.all import *
from fastai.callback.tracker import SaveModelCallback as _SaveModelCallback

In [ ]:
class EpochTracker(Callback):
    """
    Tracks epoch count, lr_max and per-epoch metrics.
    Saves to disk after every epoch for clean resume after failure.
    
    Files:
      tracker_fname — epochs_done, lr_max (for resume)
      log_fname     — full per-epoch metrics CSV
    
    Order=60 ensures this runs after Recorder(order=50) so
    recorder.values[-1] is fully populated when after_epoch fires.
    """
    order = 60

    def __init__(self, lr_max=0, total_epochs=200,
                 tracker_fname='training_stats/epoch_tracker.txt',
                 log_fname='training_stats/training_log.csv'):
        self.saved_lr_max  = lr_max
        self.total_epochs  = total_epochs
        self.tracker_fname = tracker_fname
        self.log_fname     = log_fname
        self.epochs_done, self.saved_lr_max = self._load_tracker()
        self._init_log()

    def _load_tracker(self):
        try:
            with open(self.tracker_fname, 'r') as f:
                parts = f.read().strip().split(',')
                epochs = int(parts[0])
                lr_max = float(parts[1]) if len(parts) > 1 else self.saved_lr_max
                return epochs, lr_max
        except:
            return 0, self.saved_lr_max

    def _save_tracker(self):
        os.makedirs(os.path.dirname(self.tracker_fname), exist_ok=True)
        with open(self.tracker_fname, 'w') as f:
            f.write(f"{self.epochs_done},{self.saved_lr_max}")

    def _init_log(self):
        os.makedirs(os.path.dirname(self.log_fname), exist_ok=True)
        if not Path(self.log_fname).exists():
            with open(self.log_fname, 'w', newline='') as f:
                csv.writer(f).writerow([
                    'epoch',
                    'train_loss',
                    'valid_loss',
                    'sisnr_db',
                    'noise_removed_percentage',
                    'current_lr',
                    'lr_max',
                    'epoch_elapsed_time'
                ])

    def before_epoch(self):
        self._epoch_start = time.time()
        set_epoch_seed(self.epochs_done)

    def after_epoch(self):
        """
        Runs after Recorder.after_epoch (order=50) so recorder.values[-1]
        is fully populated with [train_loss, valid_loss, sisnr_db, noise_removed_%]
        """
        elapsed  = time.time() - self._epoch_start

        # Skip ghost epochs — real epochs take at least a few seconds
        # A completed epoch at bs=2 with 48000 samples takes ~3 hours
        # Any epoch under 5 seconds is a failed/cancelled epoch
        if elapsed < 5.0:
            return

        # One-time slow_weights device verification after first real epoch
        # before increment, so this is the first epoch
        if self.epochs_done == 0:
            opt = self.learn.opt
            if hasattr(opt, 'slow_weights') and opt.slow_weights is not None:
                sw_device = opt.slow_weights[0][0].device
                print(f"[EpochTracker] slow_weights OK: {len(opt.slow_weights)} groups, "
                    f"device={sw_device}, count={opt.count}")
                if str(sw_device) == 'cpu':
                    print("[EpochTracker] WARNING: slow_weights on CPU — will crash at next sync")
            else:
                print("[EpochTracker] slow_weights=None — will rebuild from params on first step")

        self.epochs_done += 1

        recorder = self.learn.recorder

        # recorder.values[-1] layout (confirmed from debug output):
        # [train_loss, valid_loss, sisnr_db, noise_removed_%]
        vals = recorder.values[-1] if recorder.values else []

        train_loss    = vals[0] if len(vals) > 0 else None
        valid_loss    = vals[1] if len(vals) > 1 else None
        sisnr_db      = vals[2] if len(vals) > 2 else None
        noise_removed = vals[3] if len(vals) > 3 else None

        current_lr = self.learn.opt.hypers[-1]['lr'] if self.learn.opt else None

        with open(self.log_fname, 'a', newline='') as f:
            csv.writer(f).writerow([
                self.epochs_done,
                f"{train_loss:.6f}"    if train_loss     is not None else '',
                f"{valid_loss:.6f}"    if valid_loss     is not None else '',
                f"{sisnr_db:.6f}"      if sisnr_db       is not None else '',
                f"{noise_removed:.6f}" if noise_removed  is not None else '',
                f"{current_lr}"        if current_lr     is not None else '',
                f"{self.saved_lr_max}",
                f"{timedelta(seconds=elapsed)}"
            ])

        self._save_tracker()

In [ ]:
class SISNRDiagnostic(Callback):
    """
    Computes manual SI-SNR on validation batch after each epoch
    and writes to file for monitoring without interrupting training.
    """
    order = 70

    def __init__(self, fname='training_stats/sisnr_diagnostic.csv', every_n=1):
        self.fname    = fname
        self.every_n  = every_n
        self._init_file()

    def _init_file(self):
        os.makedirs(os.path.dirname(self.fname), exist_ok=True)
        if not Path(self.fname).exists():
            with open(self.fname, 'w', newline='') as f:
                csv.writer(f).writerow([
                    'epoch',
                    'sisnr_sample_0', 'sisnr_sample_1',
                    'sisnr_sample_2', 'sisnr_sample_3',
                    'sisnr_mean', 'pred_min', 'pred_max',
                    'pred_power_mean', 'targ_power_mean', 'power_ratio'
                ])

    def before_epoch(self):
        self._epoch_start = time.time()

    def after_epoch(self):
        if self._epoch_start is None:
            return
        if (time.time() - self._epoch_start) < 5.0:
            return

        epoch_num = _read_epochs_done()

        if (epoch_num) % self.every_n != 0:
            return

        try:
            self.learn.model.eval()
            xb, yb = self.learn.dls.valid.one_batch()

            with torch.no_grad():
                pred = self.learn.model(xb)

            pred_sq = pred.squeeze(1).detach().float().cpu()
            targ_sq = yb.squeeze(1).detach().float().cpu()

            targ_zm = targ_sq - targ_sq.mean(dim=-1, keepdim=True)
            pred_zm = pred_sq - pred_sq.mean(dim=-1, keepdim=True)

            eps   = 1e-8
            alpha = (targ_zm * pred_zm).sum(-1, keepdim=True) / \
                    (targ_zm.pow(2).sum(-1, keepdim=True) + eps)
            s     = (alpha * targ_zm).pow(2).sum(-1)
            n     = (pred_zm - alpha * targ_zm).pow(2).sum(-1)
            sisnr = 10 * torch.log10((s + eps) / (n + eps))

            pred_rms   = pred_sq.pow(2).mean(dim=-1).sqrt()
            targ_rms   = targ_sq.pow(2).mean(dim=-1).sqrt()
            mean_ratio = (pred_rms / (targ_rms + eps)).mean().item()

            samples = sisnr.tolist()
            while len(samples) < 4:
                samples.append('')

            with open(self.fname, 'a', newline='') as f:
                csv.writer(f).writerow([
                    epoch_num,
                    f"{samples[0]:.4f}" if samples[0] != '' else '',
                    f"{samples[1]:.4f}" if samples[1] != '' else '',
                    f"{samples[2]:.4f}" if samples[2] != '' else '',
                    f"{samples[3]:.4f}" if samples[3] != '' else '',
                    f"{sisnr.mean().item():.4f}",
                    f"{pred_sq.min().item():.4f}",
                    f"{pred_sq.max().item():.4f}",
                    f"{pred_rms.mean().item():.4f}",
                    f"{targ_rms.mean().item():.4f}",
                    f"{mean_ratio:.4f}"
                ])

        except Exception as e:
            with open(self.fname, 'a', newline='') as f:
                csv.writer(f).writerow([
                    epoch_num, f"ERROR: {e}",
                    '', '', '', '', '', '', '', '', ''
                ])
        finally:
            self.learn.model.train()

In [ ]:
class PeriodicPESQSTOI(Callback):
    """
    Computes PESQ and STOI on CPU every N epochs on a small validation subset.
    Never crashes training — all errors are caught and logged.
    
    PESQ target: > 2.5
    STOI target: > 0.88
    """
    # after EpochTracker(60) and Recorder(50)

    order = 70

    def __init__(self, fname='training_stats/pesq_stoi_log.csv',
                 every_n=10, n_batches=10):
        self.fname     = fname
        self.every_n   = every_n
        self.n_batches = n_batches
        self._init_file()

    def _init_file(self):
        os.makedirs(os.path.dirname(self.fname), exist_ok=True)
        if not Path(self.fname).exists():
            with open(self.fname, 'w', newline='') as f:
                csv.writer(f).writerow([
                    'epoch',
                    'pesq_mean', 'pesq_min', 'pesq_max',
                    'stoi_mean', 'stoi_min', 'stoi_max',
                    'n_samples', 'elapsed_seconds'
                ])

    def before_epoch(self):
        self._epoch_start = time.time()

    def after_epoch(self):
        if self._epoch_start is None:
            return
        if (time.time() - self._epoch_start) < 5.0:
            return

        epoch_num = _read_epochs_done()

        if epoch_num % self.every_n != 0:
            return

        t_start = time.time()

        try:
            from pesq import pesq as pesq_fn
            from pystoi import stoi as stoi_fn
        except ImportError as e:
            print(f"\n[PeriodicPESQSTOI] Import error: {e} — skipping")
            return

        try:
            self.learn.model.eval()
            pesq_scores = []
            stoi_scores = []

            for batch_idx, (xb, yb) in enumerate(self.learn.dls.valid):
                if batch_idx >= self.n_batches:
                    break

                with torch.no_grad():
                    pred = self.learn.model(xb)

                pred_np = pred.squeeze(1).detach().float().cpu().numpy()
                targ_np = yb.squeeze(1).detach().float().cpu().numpy()

                del pred

                for i in range(len(pred_np)):
                    try:
                        pesq_scores.append(
                            pesq_fn(16000, targ_np[i], pred_np[i], 'wb')
                        )
                    except Exception:
                        pass
                    try:
                        stoi_scores.append(
                            stoi_fn(targ_np[i], pred_np[i],
                                    16000, extended=False)
                        )
                    except Exception:
                        pass

                del pred_np, targ_np

            elapsed   = time.time() - t_start
            pesq_mean = float(np.mean(pesq_scores)) if pesq_scores else 0.0
            pesq_min  = float(np.min(pesq_scores))  if pesq_scores else 0.0
            pesq_max  = float(np.max(pesq_scores))  if pesq_scores else 0.0
            stoi_mean = float(np.mean(stoi_scores)) if stoi_scores else 0.0
            stoi_min  = float(np.min(stoi_scores))  if stoi_scores else 0.0
            stoi_max  = float(np.max(stoi_scores))  if stoi_scores else 0.0
            n_samples = len(pesq_scores)

            print(f"\n[Epoch {epoch_num}] PESQ={pesq_mean:.4f} "
                  f"(min={pesq_min:.4f} max={pesq_max:.4f}) | "
                  f"STOI={stoi_mean:.4f} "
                  f"(min={stoi_min:.4f} max={stoi_max:.4f}) | "
                  f"n={n_samples} | {elapsed:.1f}s")

            pesq_target = "✓" if pesq_mean > 2.5  else "✗"
            stoi_target = "✓" if stoi_mean > 0.88 else "✗"
            print(f"           PESQ target >2.5:  {pesq_target} | "
                  f"STOI target >0.88: {stoi_target}")

            with open(self.fname, 'a', newline='') as f:
                csv.writer(f).writerow([
                    epoch_num,
                    f"{pesq_mean:.4f}", f"{pesq_min:.4f}", f"{pesq_max:.4f}",
                    f"{stoi_mean:.4f}", f"{stoi_min:.4f}", f"{stoi_max:.4f}",
                    n_samples, f"{elapsed:.1f}"
                ])

        except Exception as e:
            elapsed = time.time() - t_start
            print(f"\n[PeriodicPESQSTOI] Error at epoch {epoch_num}: {e}")
            with open(self.fname, 'a', newline='') as f:
                csv.writer(f).writerow([
                    epoch_num, f"ERROR: {e}",
                    '', '', '', '', '', '', f"{elapsed:.1f}"
                ])

        finally:
            pesq_scores = []
            stoi_scores = []
            import gc
            gc.collect()
            self.learn.model.train()

In [ ]:
class BatchTimingCallback(Callback):
    order = 0

    def __init__(self, n_batches=100, report_every=25, 
                 out_path='training_stats/batch_timing.csv'):
        self.n_batches    = n_batches
        self.report_every = report_every
        self.out_path     = Path(out_path)
        self._rows        = []   # accumulated in memory
        self._batch_count = 0
        self._times       = defaultdict(list)

    def before_fit(self):
        self.out_path.parent.mkdir(parents=True, exist_ok=True)

    def before_batch(self):
        self._t_batch = time.perf_counter()
        self._t_phase = time.perf_counter()

    def after_pred(self):
        self._times['1_forward'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_loss(self):
        self._times['2_loss_compute'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_backward(self):
        self._times['3_backward'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_step(self):
        self._times['4_optimizer_step'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_batch(self):
        full = time.perf_counter() - self._t_batch
        self._times['0_full_batch'].append(full)
        self._batch_count += 1

        self._rows.append({
            'batch':        self._batch_count,
            'full_ms':      full * 1000,
            'forward_ms':   self._times['1_forward'][-1]     * 1000 if self._times['1_forward']     else 0,
            'loss_ms':      self._times['2_loss_compute'][-1] * 1000 if self._times['2_loss_compute'] else 0,
            'backward_ms':  self._times['3_backward'][-1]    * 1000 if self._times['3_backward']    else 0,
            'optimizer_ms': self._times['4_optimizer_step'][-1] * 1000 if self._times['4_optimizer_step'] else -1,
            # -1 means step was skipped this batch (GradientAccumulation)
        })

        if self._batch_count % self.report_every == 0:
            self._print_summary()

        if self._batch_count >= self.n_batches:
            self._print_summary()
            self._flush_csv()
            self.learn.remove_cb(self)

    def after_fit(self):
        # safety flush if training ends before n_batches reached
        if self._rows:
            self._flush_csv()

    def _flush_csv(self):
        import csv
        write_header = not self.out_path.exists()
        with open(self.out_path, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=self._rows[0].keys())
            if write_header:
                writer.writeheader()
            writer.writerows(self._rows)
        self._rows = []  # free memory after flush
        print(f"[BatchTimingCallback] flushed to {self.out_path}")

    def _print_summary(self):
        print(f"\n[BatchTimingCallback] after {self._batch_count} batches:")
        print(f"  {'Phase':<25} {'avg(ms)':>10} {'max(ms)':>10} {'total(s)':>10}")
        print(f"  {'-'*50}")
        for phase, times in sorted(self._times.items()):
            print(f"  {phase:<25} "
                  f"{np.mean(times)*1000:>10.1f} "
                  f"{np.max(times)*1000:>10.1f} "
                  f"{np.sum(times):>10.2f}")

In [ ]:
class CProfileCallback(Callback):
    """
    Runs cProfile over n_batches of training.
    
    Reports:
    1. Top functions by cumulative time
    2. Top functions by total self time  
    3. Full call chain for the single longest-running function
    4. Saves raw stats to disk for external analysis
    
    Usage:
        learn.add_cb(CProfileCallback(n_batches=50, out_dir='training_stats/cprofile'))
    """
    order = 0

    def __init__(self, n_batches=50, top_n=30, 
                 out_dir='training_stats/cprofile'):
        self.n_batches  = n_batches
        self.top_n      = top_n
        self.out_dir    = Path(out_dir)
        self._batch     = 0
        self._profiler  = None
        self._done      = False

    def before_fit(self):
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self._profiler = cProfile.Profile()
        self._profiler.enable()
        print(f"[CProfileCallback] profiling started — "
              f"will capture {self.n_batches} batches")

    def after_batch(self):
        if self._done:
            return
        self._batch += 1
        if self._batch >= self.n_batches:
            self._profiler.disable()
            self._done = True
            print(f"[CProfileCallback] {self.n_batches} batches captured - "
                  f"generating reports...")
            self._report()
            self.learn.remove_cb(self)

    def after_fit(self):
        # safety — disable if training ends before n_batches
        if not self._done and self._profiler is not None:
            self._profiler.disable()
            self._report()

    def _report(self):
        stats = pstats.Stats(self._profiler)

        # ── Save raw stats to disk for snakeviz or other tools ──────────
        raw_path = self.out_dir / 'profile.stats'
        stats.dump_stats(str(raw_path))
        print(f"[CProfileCallback] raw stats saved to {raw_path}")
        print(f"  (view with: snakeviz {raw_path}  or  "
              f"python -m pstats {raw_path})\n")

        # ── Report 1: top by cumulative time ────────────────────────────
        self._print_section(
            "TOP FUNCTIONS BY CUMULATIVE TIME "
            "(includes time in callees - best for finding slow call chains)",
            stats, sort='cumulative'
        )

        # ── Report 2: top by self time ───────────────────────────────────
        self._print_section(
            "TOP FUNCTIONS BY SELF TIME "
            "(excludes callees - best for finding actual bottleneck)",
            stats, sort='tottime'
        )

        # ── Report 3: full call chain for the top cumulative entry ───────
        self._print_call_chain(stats)

        # ── Report 4: per-phase breakdown ────────────────────────────────
        self._print_phase_breakdown(stats)

    def _print_section(self, title, stats, sort):
        s   = io.StringIO()
        ps  = pstats.Stats(self._profiler, stream=s)
        ps.sort_stats(sort)
        ps.print_stats(self.top_n)
        
        out_path = self.out_dir / f'report_{sort}.txt'
        report   = s.getvalue()
        
        with open(out_path, 'w') as f:
            f.write(f"{'='*70}\n{title}\n{'='*70}\n")
            f.write(report)
        
        print(f"\n{'='*70}")
        print(title)
        print('='*70)
        # print first 60 lines to notebook
        lines = report.split('\n')
        print('\n'.join(lines[:60]))
        if len(lines) > 60:
            print(f"  ... ({len(lines)-60} more lines in {out_path})")

    def _print_call_chain(self, stats):
        """
        Find the single function with highest cumulative time,
        then print its full caller and callee chain.
        """
        s  = io.StringIO()
        ps = pstats.Stats(self._profiler, stream=s)
        ps.sort_stats('cumulative')
        
        # get top entry from internal stats dict
        # stats.stats format: {(file,line,func): (cc, nc, tt, ct, callers)}
        #   cc=primitive calls, nc=total calls, tt=self time, ct=cumul time
        raw   = ps.stats
        top_entry = max(raw.items(), key=lambda x: x[1][3])  # sort by ct
        top_key   = top_entry[0]   # (file, line, func)
        top_data  = top_entry[1]   # (cc, nc, tt, ct, callers)

        file_, line, func = top_key
        cc, nc, tt, ct, callers = top_data

        report_lines = []
        report_lines.append(f"\n{'='*70}")
        report_lines.append("FULL CALL CHAIN FOR SLOWEST FUNCTION")
        report_lines.append('='*70)
        report_lines.append(
            f"Function : {func}\n"
            f"Location : {file_}:{line}\n"
            f"Calls    : {nc} ({cc} primitive)\n"
            f"Self time: {tt*1000/max(nc,1):.2f} ms/call  "
            f"({tt:.3f}s total)\n"
            f"Cum time : {ct*1000/max(nc,1):.2f} ms/call  "
            f"({ct:.3f}s total)"
        )

        # ── callers of top function ──────────────────────────────────────
        report_lines.append(f"\n-- CALLED BY ({len(callers)} callers) --")
        sorted_callers = sorted(
            callers.items(),
            key=lambda x: x[1][3],  # sort by cumulative time
            reverse=True
        )
        for (c_file, c_line, c_func), (c_cc, c_nc, c_tt, c_ct) in sorted_callers[:10]:
            report_lines.append(
                f"  {c_func:<45} "
                f"calls={c_nc:>6}  "
                f"cum={c_ct*1000/max(c_nc,1):>8.2f}ms/call  "
                f"@ {c_file}:{c_line}"
            )

        # ── what top function calls ──────────────────────────────────────
        report_lines.append(f"\n-- CALLS INTO --")
        callees = {
            key: data for key, data in raw.items()
            if top_key in data[4]   # data[4] is callers dict
        }
        sorted_callees = sorted(
            callees.items(),
            key=lambda x: x[1][3],
            reverse=True
        )
        for (e_file, e_line, e_func), (e_cc, e_nc, e_tt, e_ct, _) in sorted_callees[:15]:
            report_lines.append(
                f"  {e_func:<45} "
                f"calls={e_nc:>6}  "
                f"self={e_tt*1000/max(e_nc,1):>8.2f}ms/call  "
                f"cum={e_ct*1000/max(e_nc,1):>8.2f}ms/call  "
                f"@ {e_file}:{e_line}"
            )

        # ── grandcallees (one level deeper) ─────────────────────────────
        report_lines.append(f"\n-- GRANDCALLEES (next level) --")
        grandcallees = {}
        for callee_key in list(callees.keys())[:5]:  # top 5 callees only
            for key, data in raw.items():
                if callee_key in data[4]:
                    grandcallees[key] = data

        sorted_grand = sorted(
            grandcallees.items(),
            key=lambda x: x[1][3],
            reverse=True
        )
        for (g_file, g_line, g_func), (g_cc, g_nc, g_tt, g_ct, _) in sorted_grand[:15]:
            report_lines.append(
                f"  {g_func:<45} "
                f"calls={g_nc:>6}  "
                f"self={g_tt*1000/max(g_nc,1):>8.2f}ms/call  "
                f"cum={g_ct*1000/max(g_nc,1):>8.2f}ms/call  "
                f"@ {g_file}:{g_line}"
            )

        report = '\n'.join(report_lines)
        out_path = self.out_dir / 'call_chain.txt'
        with open(out_path, 'w') as f:
            f.write(report)
        print(report)
        print(f"\n  (saved to {out_path})")

    def _print_phase_breakdown(self, stats):
        """
        Aggregate time by module/phase — fastai, torch, model, loss, etc.
        Groups entries by filename prefix for a high-level view.
        """
        raw = stats.stats
        
        phase_totals = {}
        for (file_, line, func), (cc, nc, tt, ct, callers) in raw.items():
            # categorize by file path
            if 'fastai'        in file_: phase = 'fastai'
            elif 'torch'       in file_: phase = 'torch'
            elif 'directml'    in file_: phase = 'directml'
            elif 'ipykernel'   in file_: phase = 'your_code (ipykernel)'
            elif 'pyroomacous' in file_: phase = 'pyroomacoustics'
            elif 'numpy'       in file_: phase = 'numpy'
            elif 'scipy'       in file_: phase = 'scipy'
            elif '<'           in file_: phase = 'builtins/C_extensions'
            else:                        phase = f'other: {Path(file_).name}'

            if phase not in phase_totals:
                phase_totals[phase] = {'tt': 0, 'ct': 0, 'calls': 0}
            phase_totals[phase]['tt']    += tt
            phase_totals[phase]['ct']    += ct
            phase_totals[phase]['calls'] += nc

        report_lines = [
            f"\n{'='*70}",
            "TIME BY MODULE/PHASE",
            '='*70,
            f"{'Module':<35} {'self(s)':>10} {'cum(s)':>10} {'calls':>10}",
            '-'*65
        ]

        for phase, data in sorted(
            phase_totals.items(), 
            key=lambda x: -x[1]['tt']
        ):
            report_lines.append(
                f"{phase:<35} "
                f"{data['tt']:>10.3f} "
                f"{data['ct']:>10.3f} "
                f"{data['calls']:>10,}"
            )

        report = '\n'.join(report_lines)
        out_path = self.out_dir / 'phase_breakdown.txt'
        with open(out_path, 'w') as f:
            f.write(report)
        print(report)
        print(f"\n  (saved to {out_path})")

In [ ]:
class RecorderCleaner(Callback):
    order = 80
    def after_epoch(self):
        if len(self.learn.recorder.log) < 3:
            return
        r = self.learn.recorder
        n_losses = len(r.losses)
        n_lrs    = len(r.lrs)
        n_iters  = len(r.iters)
        r.losses.clear()
        r.lrs.clear()
        # Keep iters in sync with losses/lrs — clear all but append
        # a sentinel 0 so smooth_loss.count is still readable if needed
        r.iters.clear()
        print(f"[RecorderCleaner] epoch {self.learn.epoch} — "
              f"cleared losses={n_losses}  lrs={n_lrs}  iters={n_iters}")

    def after_fit(self):
        r = self.learn.recorder
        r.losses.clear()
        r.lrs.clear()
        r.iters.clear()

In [ ]:
class DMLQueueFlush(Callback):
    order = 99  # after everything else
    def after_epoch(self):
        # Force DML to flush its command queue — resets queue depth
        # so next epoch starts clean rather than inheriting accumulated depth
        try:
            _ = torch.tensor(1.0, device=device).item()  # force sync
        except Exception:
            pass

In [ ]:
def _read_epochs_done():
    """Read current epochs_done from tracker file"""
    try:
        with open('training_stats/epoch_tracker.txt', 'r') as f:
            return int(f.read().strip().split(',')[0])
    except:
        return 0

In [ ]:
def set_epoch_seed(epoch):
    """Call before each epoch from main process"""
    EPOCH_SEED.value = epoch
    try:
        torch_directml.PrivateUse1Module.manual_seed_all(epoch)
    except Exception:
        pass